# VAE на реальных импульсах 60Co и 252Cf

Этот ноутбук воспроизводит идею трёхмерного VAE из отчёта коллеги на реальных данных PSD_ML. Пять моделей обучаются отдельно по каналам **только на безметочной смеси 252Cf**. События 60Co не участвуют в оптимизации и используются как внешний гамма-контроль запуска.

Все преобразования данных и обучение реализованы в `psd_ml.vae`; здесь остаются параметры, вызовы этапов, визуализации и интерпретация.

## Методические ограничения

- В наличии только один Co- и один Cf-запуск, поэтому это исследование латентной структуры, а не независимый тест neutron/gamma-классификатора.
- Цвет Co/Cf означает происхождение события из запуска, а не истинную event-level метку частицы.
- Три латентные переменные исследуются равноправно. Номер координаты, её знак и порядок могут меняться между seed.
- `low_snr` сохраняется в анализе; удаляются только структурные дефекты формы.

In [1]:
from psd_ml.pipeline import configure_plotly, discover_project
from psd_ml.vae import (
    RealVAEConfig,
    audit_real_vae_latents,
    plot_real_vae_results,
    prepare_real_vae_data,
    print_real_vae_data_summary,
    print_real_vae_findings,
    train_channel_vae_ensemble,
)

configure_plotly()
paths = discover_project()
config = RealVAEConfig(
    data_seed=20260717,
    pulses_per_group=10_000,
    model_seeds=(20260717, 20260718, 20260719),
    latent_dim=3,
    max_epochs=200,
    patience=20,
)
config

RealVAEConfig(data_seed=20260717, pulses_per_group=10000, model_seeds=(20260717, 20260718, 20260719), latent_dim=3, hidden_dims=(128, 64), validation_fraction=0.2, batch_size=256, learning_rate=0.001, max_epochs=200, patience=20, beta_max=0.01, beta_warmup_epochs=40, tail_start=40, integration_start=15, integration_stop=100, qlong_bins=8, traversal_quantiles=(0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99))

## 1. Подготовка 100 000 реальных импульсов

Выбираются 10 000 событий из каждой пары `run × channel`. Используется та же цепочка baseline subtraction, полярности, CFD-50-выравнивания и peak-нормировки, что и в основном ноутбуке. Для всех sampled событий проверяется полное совпадение 144 CSV- и ROOT-отсчётов, после чего присоединяются `Qlong` и `Qshort`.

In [2]:
vae_data = prepare_real_vae_data(
    paths,
    config,
    verify_root_waveforms=True,
)
data_summary = print_real_vae_data_summary(vae_data)

VAE учится только на Cf; Co используется только как внешний run-control.
CH0 PMT-9102B + T-Stlbn: train=7842, validation=1962, Co-control=10000, excluded=196, low-SNR retained=1
CH2 PMT-9102B + T-Stlbn: train=7829, validation=1957, Co-control=10000, excluded=214, low-SNR retained=3
CH3 PMT-R6094 + P-Trfnl: train=7933, validation=1983, Co-control=10000, excluded=84, low-SNR retained=433
CH4 PMT-R6231 + T-Stlbn: train=7900, validation=1975, Co-control=9998, excluded=127, low-SNR retained=745
CH5 PMT-R6231 + P-Trfnl: train=7997, validation=2000, Co-control=9997, excluded=6, low-SNR retained=14


## 2. Обучение пяти channel-specific VAE

Для каждого канала используется один и тот же Cf train/validation split, стратифицированный по децилям `Qlong`. Три seed меняют только инициализацию и порядок обучающих batch. Архитектура фиксирована: `144 → 128 → 64 → (mu[3], logvar[3]) → 64 → 128 → 144`.

In [3]:
vae_ensemble = train_channel_vae_ensemble(
    vae_data,
    paths,
    config,
)

CH0, seed=20260717: best epoch=190, validation loss=0.144972


CH0, seed=20260718: best epoch=175, validation loss=0.146506


CH0, seed=20260719: best epoch=95, validation loss=0.179440


CH2, seed=20260717: best epoch=198, validation loss=0.146941


CH2, seed=20260718: best epoch=28, validation loss=0.166406


...

CH4, seed=20260718: best epoch=96, validation loss=0.108808


CH4, seed=20260719: best epoch=193, validation loss=0.107761


CH5, seed=20260717: best epoch=30, validation loss=0.038877


CH5, seed=20260718: best epoch=55, validation loss=0.039619


CH5, seed=20260719: best epoch=83, validation loss=0.039542


## 3. Полный latent audit

В анализе используются детерминированные posterior means `mu`, а не случайные выборки `z`. Каждая из трёх координат сопоставляется с `Qlong`, классическим PSD, ручным `shape_score`, амплитудой, SNR, baseline и CFD shift. Латенты разных seed согласуются только перестановкой координат и знаком на общей Cf-validation выборке.

In [4]:
latent_audit = audit_real_vae_latents(
    vae_data,
    vae_ensemble,
    paths,
    config,
)

## 4. Интерактивные визуализации

Для каждого канала строятся loss-кривые, реальные и восстановленные медианные формы, распределения всех трёх координат, 3D-латентное пространство, зависимости внутри `Qlong`-страт и traversal каждой координаты по квантилям реально наблюдаемого posterior. Все фигуры дополнительно сохраняются как `.html` в `gamma_n_data/samples/vae_real/`.

In [5]:
vae_figures = plot_real_vae_results(
    vae_data,
    vae_ensemble,
    latent_audit,
    config,
)

## 5. Численная сводка и правила интерпретации

`var(mu)` и средний KL показывают, использует ли VAE координату. Co-vs-Cf run AUC показывает только различимость двух экспериментальных запусков. Он не является neutron/gamma AUC и не используется как доказательство физической классификации. Даже если несколько координат меняют форму, все они сохраняются для следующего общего PSD benchmark.

In [6]:
print_real_vae_findings(latent_audit, config)

Все три латентные переменные сохранены; координаты не считаются физическими метками.
Согласование seed выполнено только по перестановке и знаку на Cf-validation.
CH0 PMT-9102B + T-Stlbn:
  z1: var(mu)=0.7331, mean KL=0.7507, Co-vs-Cf run AUC=0.527
  z2: var(mu)=0.8308, mean KL=0.945, Co-vs-Cf run AUC=0.517
  z3: var(mu)=0.958, mean KL=1.673, Co-vs-Cf run AUC=0.620
CH2 PMT-9102B + T-Stlbn:
  z1: var(mu)=0.7884, mean KL=0.7879, Co-vs-Cf run AUC=0.548
  z2: var(mu)=0.939, mean KL=1.706, Co-vs-Cf run AUC=0.614
  z3: var(mu)=0.8039, mean KL=0.9495, Co-vs-Cf run AUC=0.515
CH3 PMT-R6094 + P-Trfnl:
  z1: var(mu)=0.01954, mean KL=0.03023, Co-vs-Cf run AUC=0.508
  z2: var(mu)=0.01157, mean KL=0.01446, Co-vs-Cf run AUC=0.501
  z3: var(mu)=0.4718, mean KL=1.375, Co-vs-Cf run AUC=0.507
CH4 PMT-R6231 + T-Stlbn:
  z1: var(mu)=0.001836, mean KL=0.002332, Co-vs-Cf run AUC=0.607
  z2: var(mu)=0.5763, mean KL=0.4268, Co-vs-Cf run AUC=0.617
  z3: var(mu)=0.8033, mean KL=1.293, Co-vs-Cf run AUC=0.561
CH5 P

## Как воспроизвести

Из корня проекта:

```bash
.venv/bin/python -m pip install -r requirements-vae.txt
JUPYTER_PATH="$PWD/.jupyter" .venv/bin/python -m jupyter nbconvert --to notebook --execute --inplace real_data_vae.ipynb --ExecutePreprocessor.timeout=7200
```

Модели, таблицы латентов и HTML являются воспроизводимыми производными данными и не включаются в Git.